# XGBoost vs Ridge Regression — прогноз RPS на 5 минут (per handler)

**Датасет:** временные ряды RPS по handler-ам (`/accounts/`, `/transfers/`, `/users/`), шаг 15 секунд  
**Горизонт прогноза:** 5 минут = 20 шагов вперёд  
**Подход:** lag/rolling/EWM признаки + временны́е циклические признаки; отдельная модель на каждый handler

## 1. Импорты и параметры

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

np.random.seed(42)

# ── Параметры задачи ────────────────────────────────────────────
FREQ_SEC   = 15          # интервал между точками, с
HORIZON    = 20          # горизонт прогноза (5 мин = 20 × 15 с)

LAGS         = [1, 2, 4, 8, 16, 32, 60, 120, 240]
ROLL_WINDOWS = [4, 20, 60, 120, 240]
EWM_SPANS    = [4, 20, 60]

RPS_FILES = [
    '../data/accounts__rps.csv',
    '../data/transfers__rps.csv',
    '../data/users__rps.csv',
]

print('✅ Окружение готово')
print(f'   Горизонт: {HORIZON} шагов = {HORIZON * FREQ_SEC // 60} мин')
print(f'   Лаги: {LAGS}')
print(f'   Rolling windows: {ROLL_WINDOWS}')
print(f'   EWM spans: {EWM_SPANS}')

## 2. Загрузка и подготовка данных

In [ ]:
dfs = []
for f in RPS_FILES:
    if os.path.exists(f):
        d = pd.read_csv(f)
        dfs.append(d)
        print(f'Loaded {f}: {d.shape}')
    else:
        print(f'⚠️  Файл не найден: {f}')

raw = pd.concat(dfs, ignore_index=True)
raw['datetime'] = pd.to_datetime(raw['timestamp'], unit='s')
raw = raw.sort_values(['handler', 'datetime']).reset_index(drop=True)

HANDLERS = sorted(raw['handler'].unique().tolist())
print(f'\nHandlers: {HANDLERS}')

# Каждый handler — собственный непрерывный ряд с шагом 15 с
series_by_handler: dict[str, pd.Series] = {}
for h in HANDLERS:
    sub = raw[raw['handler'] == h].drop_duplicates('timestamp')
    sub = sub.set_index('datetime')['value']
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq='15s')
    sub = sub.reindex(full_idx).interpolate('time').fillna(0.0).clip(lower=0.0)
    sub.index.name = 'datetime'
    series_by_handler[h] = sub
    print(f'  {h:32s}  rows={len(sub):>7}  mean={sub.mean():.3f}  max={sub.max():.3f}  std={sub.std():.3f}')

## 3. Разведочный анализ (EDA)

In [ ]:
fig, axes = plt.subplots(len(HANDLERS), 2, figsize=(18, 4 * len(HANDLERS)))
fig.suptitle('RPS по handler-ам: полный ряд и распределение', fontsize=14, fontweight='bold')

colors = ['steelblue', 'tomato', 'seagreen']

for i, (h, color) in enumerate(zip(HANDLERS, colors)):
    s = series_by_handler[h]

    # Полный ряд
    ax = axes[i, 0]
    ax.plot(s.index, s.values, lw=0.4, color=color, alpha=0.8)
    ax.set_title(f'{h} — временной ряд', fontsize=11)
    ax.set_ylabel('RPS')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.grid(alpha=0.3)

    # Гистограмма
    ax = axes[i, 1]
    ax.hist(s.values, bins=60, color=color, edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.set_title(f'{h} — распределение RPS', fontsize=11)
    ax.set_xlabel('RPS')
    ax.set_ylabel('Частота')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Автокорреляция по handler-ам

In [ ]:
fig, axes = plt.subplots(1, len(HANDLERS), figsize=(18, 4), sharey=False)
fig.suptitle('Автокорреляция RPS (лаги 1–120)', fontsize=13, fontweight='bold')

MAX_LAG = 120
colors = ['steelblue', 'tomato', 'seagreen']

for ax, (h, color) in zip(axes, zip(HANDLERS, colors)):
    vals = series_by_handler[h].values
    acf = [np.corrcoef(vals[:-k], vals[k:])[0, 1] for k in range(1, MAX_LAG + 1)]
    ax.bar(range(1, MAX_LAG + 1), acf, color=color, alpha=0.7, width=0.8)
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(HORIZON, color='red', ls='--', lw=1.5, label=f'горизонт ({HORIZON})')
    ax.set_title(h, fontsize=10)
    ax.set_xlabel('Лаг (шаги по 15 с)')
    ax.set_ylabel('Корреляция')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Конструирование признаков

In [ ]:
def make_features(series: pd.Series) -> pd.DataFrame:
    """Строит матрицу признаков из временного ряда RPS."""
    f = pd.DataFrame(index=series.index)

    # Лаговые признаки
    for lag in LAGS:
        f[f'lag_{lag}'] = series.shift(lag)

    # Скользящие статистики
    for w in ROLL_WINDOWS:
        rolled = series.shift(1).rolling(w)
        f[f'roll_mean_{w}'] = rolled.mean()
        f[f'roll_std_{w}']  = rolled.std()

    # EWM
    for span in EWM_SPANS:
        f[f'ewm_{span}'] = series.shift(1).ewm(span=span).mean()

    # Временны́е циклические признаки
    f['hour_sin']   = np.sin(2 * np.pi * series.index.hour / 24)
    f['hour_cos']   = np.cos(2 * np.pi * series.index.hour / 24)
    f['minute_sin'] = np.sin(2 * np.pi * series.index.minute / 60)
    f['minute_cos'] = np.cos(2 * np.pi * series.index.minute / 60)

    # Разности (тренд)
    f['diff_1']  = series.diff(1).shift(1)
    f['diff_20'] = series.diff(20).shift(1)

    return f


# Проверка на одном handler-е
sample_h = HANDLERS[0]
sample_X = make_features(series_by_handler[sample_h])
print(f'Признаков: {sample_X.shape[1]}')
print(f'Список: {list(sample_X.columns)}')

## 6. Train / Val / Test split (временной, 70/15/15)

In [ ]:
datasets: dict[str, dict] = {}

for h in HANDLERS:
    s = series_by_handler[h]
    X = make_features(s)
    y = s.shift(-HORIZON).rename('target')

    valid_idx = X.dropna().index.intersection(y.dropna().index)
    X, y = X.loc[valid_idx], y.loc[valid_idx]

    n = len(X)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.85)

    X_train, y_train = X.iloc[:train_end],       y.iloc[:train_end]
    X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
    X_test,  y_test  = X.iloc[val_end:],          y.iloc[val_end:]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)

    datasets[h] = dict(
        X_train=X_train, y_train=y_train,
        X_val=X_val,     y_val=y_val,
        X_test=X_test,   y_test=y_test,
        X_train_s=X_train_s, X_val_s=X_val_s, X_test_s=X_test_s,
        scaler=scaler,
        feature_cols=list(X.columns),
        datetime_test=s.loc[X_test.index].index,
    )
    print(f'{h:32s}  train={len(X_train):>6,}  val={len(X_val):>5,}  test={len(X_test):>5,}')

## 7. Ridge Regression (per handler)

In [ ]:
ridge_models: dict[str, Ridge] = {}
ridge_preds:  dict[str, dict]  = {}

alphas = np.logspace(-2, 4, 30)

for h in HANDLERS:
    d = datasets[h]
    ridge_cv = RidgeCV(alphas=alphas, cv=5)
    ridge_cv.fit(d['X_train_s'], d['y_train'])

    ridge = Ridge(alpha=ridge_cv.alpha_)
    ridge.fit(d['X_train_s'], d['y_train'])

    ridge_models[h] = ridge
    ridge_preds[h] = {
        'val':  ridge.predict(d['X_val_s']),
        'test': ridge.predict(d['X_test_s']),
    }

    mae_v = mean_absolute_error(d['y_val'],  ridge_preds[h]['val'])
    mae_t = mean_absolute_error(d['y_test'], ridge_preds[h]['test'])
    print(f'{h:32s}  alpha={ridge_cv.alpha_:8.4f}  MAE val={mae_v:.4f}  MAE test={mae_t:.4f}')

## 8. XGBoost (per handler)

In [ ]:
xgb_models: dict[str, XGBRegressor] = {}
xgb_preds:  dict[str, dict]          = {}

for h in HANDLERS:
    d = datasets[h]
    xgb = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        early_stopping_rounds=30,
        eval_metric='rmse',
        random_state=42,
        verbosity=0,
    )
    xgb.fit(
        d['X_train'], d['y_train'],
        eval_set=[(d['X_val'], d['y_val'])],
        verbose=False,
    )
    xgb_models[h] = xgb
    xgb_preds[h] = {
        'val':  xgb.predict(d['X_val']),
        'test': xgb.predict(d['X_test']),
    }

    mae_v = mean_absolute_error(d['y_val'],  xgb_preds[h]['val'])
    mae_t = mean_absolute_error(d['y_test'], xgb_preds[h]['test'])
    print(f'{h:32s}  best_iter={xgb.best_iteration:>4}  MAE val={mae_v:.4f}  MAE test={mae_t:.4f}')

## 9. Сравнение метрик

In [ ]:
def mape(y_true, y_pred, eps=1e-6):
    return np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-9) * 100

def compute_metrics(y_true, y_pred, model_name, handler, split):
    yt = np.array(y_true)
    return {
        'Handler': handler,
        'Модель': model_name,
        'Split': split,
        'MAE':     round(mean_absolute_error(yt, y_pred), 4),
        'MSE':     round(mean_squared_error(yt, y_pred),  4),
        'MAPE, %': round(mape(yt, y_pred), 4),
        'WAPE, %': round(wape(yt, y_pred), 4),
    }

rows = []
for h in HANDLERS:
    d = datasets[h]
    for split, y_true in [('Val', d['y_val']), ('Test', d['y_test'])]:
        rows.append(compute_metrics(y_true, ridge_preds[h][split.lower()], 'Ridge',   h, split))
        rows.append(compute_metrics(y_true, xgb_preds[h][split.lower()],   'XGBoost', h, split))

results = pd.DataFrame(rows)
results_test = results[results['Split'] == 'Test'].set_index(['Handler', 'Модель']).drop(columns='Split')

print('=== Test set ===')
display(results_test.style
    .highlight_min(color='#d4edda', subset=['MAE', 'MSE', 'MAPE, %', 'WAPE, %'])
    .format(precision=4))

## 10. Визуализация метрик по handler-ам

In [ ]:
metrics_list = ['MAE', 'MSE', 'MAPE, %', 'WAPE, %']
n_handlers = len(HANDLERS)
n_metrics  = len(metrics_list)

fig, axes = plt.subplots(n_handlers, n_metrics, figsize=(5 * n_metrics, 4 * n_handlers))
fig.suptitle('Сравнение метрик: XGBoost vs Ridge (Test set)', fontsize=14, fontweight='bold')

bar_colors = {'Ridge': '#5b8db8', 'XGBoost': '#e07b54'}

for row_i, h in enumerate(HANDLERS):
    sub = results_test.loc[h]
    for col_i, metric in enumerate(metrics_list):
        ax = axes[row_i, col_i]
        vals   = [sub.loc['Ridge', metric], sub.loc['XGBoost', metric]]
        labels = ['Ridge', 'XGBoost']
        bars = ax.bar(labels, vals, color=[bar_colors[l] for l in labels], width=0.5, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_title(f'{h}\n{metric}', fontsize=9)
        ax.set_ylim(0, max(vals) * 1.35 + 1e-9)
        ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Визуализация прогнозов на тестовой выборке

In [ ]:
SHOW = 6 * 60 * 4   # 6 часов × 4 точки/мин

fig, axes = plt.subplots(len(HANDLERS) * 2, 1, figsize=(18, 5 * len(HANDLERS) * 2), sharex=False)
fig.suptitle('Прогноз RPS через 5 минут (фрагмент тестовой выборки)', fontsize=14, fontweight='bold')

ax_idx = 0
for h in HANDLERS:
    d = datasets[h]
    dt = d['datetime_test']
    sl = slice(None, SHOW)

    for pred, label, color in [
        (ridge_preds[h]['test'], 'Ridge',   '#5b8db8'),
        (xgb_preds[h]['test'],   'XGBoost', '#e07b54'),
    ]:
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(dt[sl], d['y_test'].values[sl],
                lw=1.2, color='#333', label='Факт', alpha=0.85)
        ax.plot(dt[sl], pred[sl],
                lw=1.0, color=color, ls='--', label=f'{label} (+5 мин)', alpha=0.9)
        mae_s = mean_absolute_error(d['y_test'].values[sl], pred[sl])
        ax.set_title(f'{h}  |  {label}  |  MAE = {mae_s:.4f} RPS', fontsize=10)
        ax.set_ylabel('RPS')
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))

plt.tight_layout()
plt.show()

## 12. Анализ остатков

In [ ]:
fig, axes = plt.subplots(len(HANDLERS) * 2, 2, figsize=(14, 5 * len(HANDLERS) * 2))
fig.suptitle('Анализ остатков (Test set)', fontsize=13, fontweight='bold')

ax_row = 0
for h in HANDLERS:
    d = datasets[h]
    for pred, label, color in [
        (ridge_preds[h]['test'], 'Ridge',   '#5b8db8'),
        (xgb_preds[h]['test'],   'XGBoost', '#e07b54'),
    ]:
        res = d['y_test'].values - pred

        # Остатки vs предсказанное
        ax = axes[ax_row, 0]
        ax.scatter(pred, res, alpha=0.12, s=3, color=color)
        ax.axhline(0, color='k', lw=1)
        ax.set_title(f'{h} / {label}: остатки vs предсказанное', fontsize=9)
        ax.set_xlabel('Предсказание, RPS')
        ax.set_ylabel('Остаток, RPS')
        ax.grid(alpha=0.3)

        # Гистограмма остатков
        ax = axes[ax_row, 1]
        ax.hist(res, bins=80, color=color, edgecolor='white', lw=0.3, alpha=0.85)
        ax.axvline(0, color='k', lw=1.2)
        ax.axvline(res.mean(), color='red', lw=1.5, ls='--', label=f'mean={res.mean():.4f}')
        ax.set_title(f'{h} / {label}: распределение остатков', fontsize=9)
        ax.set_xlabel('Остаток, RPS')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        ax_row += 1

plt.tight_layout()
plt.show()

## 13. Важность признаков XGBoost (per handler)

In [ ]:
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor='#e07b54', label='Лаговые (lag)'),
    Patch(facecolor='#5b8db8', label='Скользящие (roll/ewm)'),
    Patch(facecolor='#76b876', label='Временны́е / разности'),
]

fig, axes = plt.subplots(1, len(HANDLERS), figsize=(7 * len(HANDLERS), 7))
fig.suptitle('XGBoost: топ-20 признаков по важности (per handler)', fontsize=13, fontweight='bold')

for ax, h in zip(axes, HANDLERS):
    xgb_m = xgb_models[h]
    feat_imp = pd.Series(xgb_m.feature_importances_, index=datasets[h]['feature_cols'])
    top20 = feat_imp.nlargest(20)
    bar_c = [
        '#e07b54' if 'lag' in n
        else '#5b8db8' if ('roll' in n or 'ewm' in n)
        else '#76b876'
        for n in top20.index
    ]
    top20.sort_values().plot(kind='barh', ax=ax, color=bar_c[::-1])
    ax.set_title(h, fontsize=10)
    ax.set_xlabel('Feature importance (gain)')
    ax.grid(alpha=0.3, axis='x')
    ax.legend(handles=legend_elements, loc='lower right', fontsize=7)

plt.tight_layout()
plt.show()

## 14. Коэффициенты Ridge (топ-20, per handler)

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, len(HANDLERS), figsize=(7 * len(HANDLERS), 7))
fig.suptitle('Ridge: топ-20 признаков по |коэффициенту| (per handler)', fontsize=13, fontweight='bold')

for ax, h in zip(axes, HANDLERS):
    ridge_m = ridge_models[h]
    coef = pd.Series(np.abs(ridge_m.coef_), index=datasets[h]['feature_cols'])
    top20 = coef.nlargest(20)
    bar_c = [
        '#e07b54' if 'lag' in n
        else '#5b8db8' if ('roll' in n or 'ewm' in n)
        else '#76b876'
        for n in top20.index
    ]
    top20.sort_values().plot(kind='barh', ax=ax, color=bar_c[::-1])
    ax.set_title(h, fontsize=10)
    ax.set_xlabel('|Коэффициент|')
    ax.grid(alpha=0.3, axis='x')
    ax.legend(handles=legend_elements, loc='lower right', fontsize=7)

plt.tight_layout()
plt.show()

## 15. Итоговый вывод

In [ ]:
print('=' * 70)
print('         ИТОГОВОЕ СРАВНЕНИЕ (Test set, все handler-ы)')
print('=' * 70)
print(results_test.to_string())
print()

for h in HANDLERS:
    sub = results_test.loc[h]
    ridge_mae = sub.loc['Ridge',   'MAE']
    xgb_mae   = sub.loc['XGBoost', 'MAE']
    winner    = 'XGBoost' if xgb_mae < ridge_mae else 'Ridge'
    diff_pct  = abs(ridge_mae - xgb_mae) / max(ridge_mae, xgb_mae) * 100
    print(f'{h:32s}  🏆 {winner:7s}  Δ MAE = {diff_pct:.1f}%')

print()
print(f'Горизонт прогноза : {HORIZON} шагов = {HORIZON * FREQ_SEC // 60} мин')
print(f'Лаги              : {LAGS}')
print(f'Rolling windows   : {ROLL_WINDOWS}')
print(f'EWM spans         : {EWM_SPANS}')